# FedMed-BN: Notebook 3 - Federated Learning (FedAvg) + DP
**Simulation mode on Google Colab using Flower framework**

In [ ]:
# Cell 1: Install & Imports
!pip install flwr opacus -q

import flwr as fl
from flwr.simulation import start_simulation
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, AdamW
from collections import OrderedDict
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import json
import numpy as np
from tqdm.auto import tqdm
import os
import pickle

os.makedirs('/content/FedMed-BN/models/federated', exist_ok=True)
os.makedirs('/content/FedMed-BN/models/federated_dp', exist_ok=True)
os.makedirs('/content/FedMed-BN/results', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# Cell 2: Load data & label mappings
MODEL_NAME = "csebuetnlp/banglabert"  # Full model (110M). Use banglabert_small on Colab free tier.

with open('/content/FedMed-BN/data/tag2id.json', 'r') as f:
    tag2id = json.load(f)
with open('/content/FedMed-BN/data/id2tag.json', 'r') as f:
    id2tag = {int(k): v for k, v in json.load(f).items()}

def load_bio_file(filepath):
    sentences = []
    current = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current.append((parts[0], parts[1]))
    if current:
        sentences.append(current)
    return sentences

def tokenize_and_align_labels(sentences, tokenizer, max_length=128):
    tokenized_inputs = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for tokens, tags in sentences:
        encoding = tokenizer(tokens, is_split_into_words=True, truncation=True,
                           padding='max_length', max_length=max_length, return_tensors=None)
        word_ids = encoding.word_ids()
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id.get(tags[word_idx], tag2id['O']))
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        tokenized_inputs['input_ids'].append(encoding['input_ids'])
        tokenized_inputs['attention_mask'].append(encoding['attention_mask'])
        tokenized_inputs['labels'].append(label_ids)
    return tokenized_inputs

class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.encodings['labels'][idx])
        }
    def __len__(self):
        return len(self.encodings['input_ids'])

# Load client data
client_train_loaders = []
client_test_loaders = []

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

for i in range(3):
    train_sents = load_bio_file(f'/content/FedMed-BN/data/client_{i}_train.txt')
    test_sents = load_bio_file(f'/content/FedMed-BN/data/client_{i}_test.txt')
    
    train_enc = tokenize_and_align_labels(train_sents, tokenizer)
    test_enc = tokenize_and_align_labels(test_sents, tokenizer)
    
    train_loader = DataLoader(NERDataset(train_enc), batch_size=8, shuffle=True, drop_last=True)
    test_loader = DataLoader(NERDataset(test_enc), batch_size=8, shuffle=False, drop_last=True)
    
    client_train_loaders.append(train_loader)
    client_test_loaders.append(test_loader)
    
    print(f"Client {i}: train batches={len(train_loader)}, test batches={len(test_loader)}")

In [ ]:
# Cell 3: Flower Client (without DP)
num_labels = len(tag2id)
LOCAL_EPOCHS = 3

class BanglaBERTClient(fl.client.NumPyClient):
    def __init__(self, client_id, train_loader, test_loader, device):
        self.client_id = client_id
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        # Each client gets a FRESH model copy
        self.model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2tag, label2id=tag2id
        ).to(device)
    
    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]
    
    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)
    
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        optimizer = AdamW(self.model.parameters(), lr=2e-5)
        
        for epoch in range(LOCAL_EPOCHS):
            for batch in self.train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
        
        return self.get_parameters(config={}), len(self.train_loader.dataset), {}
    
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        
        all_preds = []
        all_labels = []
        total_loss = 0
        
        with torch.no_grad():
            for batch in self.test_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                predictions = outputs.logits.argmax(dim=-1)
                
                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Compute metrics
        true_predictions = [
            [id2tag[p] for (p, l) in zip(pred, label) if l != -100]
            for pred, label in zip(all_preds, all_labels)
        ]
        true_labels = [
            [id2tag[l] for (p, l) in zip(pred, label) if l != -100]
            for pred, label in zip(all_preds, all_labels)
        ]
        
        accuracy = sum(p == l for pred, label in zip(true_predictions, true_labels)
                       for p, l in zip(pred, label)) / sum(len(l) for l in true_labels)
        f1 = f1_score(true_labels, true_predictions)
        
        return float(total_loss / len(self.test_loader)), len(self.test_loader.dataset),
               {"accuracy": accuracy, "f1": f1}

In [ ]:
# Cell 4: Run Federated Simulation (FedAvg, NO DP)
def client_fn(cid: str):
    client_id = int(cid)
    return BanglaBERTClient(
        client_id,
        client_train_loaders[client_id],
        client_test_loaders[client_id],
        device
    )

# FedAvg Strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=3,
    min_evaluate_clients=3,
    min_available_clients=3,
)

NUM_ROUNDS = 10

print(f"Starting FedAvg simulation for {NUM_ROUNDS} rounds with 3 clients...")

history = start_simulation(
    client_fn=client_fn,
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources={"num_gpus": 1.0, "num_cpus": 2.0},
)

In [ ]:
# Cell 5: Plot & Save Results (FedAvg No DP)
import matplotlib.pyplot as plt

# Extract metrics from history
rounds = range(1, NUM_ROUNDS + 1)
centralized_loss = history.losses_distributed
centralized_acc = history.metrics_distributed.get('accuracy', [])
centralized_f1 = history.metrics_distributed.get('f1', [])

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
if centralized_loss:
    plt.plot([r for r, _ in centralized_loss], [l for _, l in centralized_loss], 'o-')
plt.title('FedAvg: Loss per Round')
plt.xlabel('Round')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
if centralized_acc:
    plt.plot([r for r, _ in centralized_acc], [a for _, a in centralized_acc], 'o-')
plt.title('FedAvg: Accuracy per Round')
plt.xlabel('Round')
plt.ylabel('Accuracy')

plt.subplot(1, 3, 3)
if centralized_f1:
    plt.plot([r for r, _ in centralized_f1], [f for _, f in centralized_f1], 'o-')
plt.title('FedAvg: F1 per Round')
plt.xlabel('Round')
plt.ylabel('F1')

plt.tight_layout()
plt.savefig('/content/FedMed-BN/results/fedavg_no_dp_curves.png')
plt.show()

# Save history
with open('/content/FedMed-BN/results/fedavg_no_dp_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print("FedAvg (no DP) complete!")

In [ ]:
# Cell 6: Flower Client WITH Differential Privacy (Opacus)
from opacus import PrivacyEngine

class DPBanglaBERTClient(fl.client.NumPyClient):
    def __init__(self, client_id, train_loader, test_loader, device,
                 target_epsilon=3.0, target_delta=1e-5,
                 max_grad_norm=1.0, noise_multiplier=1.1):
        self.client_id = client_id
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.target_epsilon = target_epsilon
        self.target_delta = target_delta
        self.max_grad_norm = max_grad_norm
        self.noise_multiplier = noise_multiplier
        
        self.model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, id2label=id2tag, label2id=tag2id
        ).to(device)
    
    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]
    
    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)
    
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        optimizer = AdamW(self.model.parameters(), lr=2e-5)
        
        # Wrap with Opacus PrivacyEngine
        privacy_engine = PrivacyEngine()
        model, optimizer, train_loader = privacy_engine.make_private(
            module=self.model,
            optimizer=optimizer,
            data_loader=self.train_loader,
            noise_multiplier=self.noise_multiplier,
            max_grad_norm=self.max_grad_norm,
        )
        
        for epoch in range(LOCAL_EPOCHS):
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
        
        # Get actual epsilon spent
        epsilon = privacy_engine.get_epsilon(delta=self.target_delta)
        print(f"Client {self.client_id} epsilon: {epsilon:.2f}")
        
        return self.get_parameters(config={}), len(self.train_loader.dataset), {"epsilon": epsilon}
    
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        
        all_preds = []
        all_labels = []
        total_loss = 0
        
        with torch.no_grad():
            for batch in self.test_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_loss += outputs.loss.item()
                predictions = outputs.logits.argmax(dim=-1)
                
                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        true_predictions = [
            [id2tag[p] for (p, l) in zip(pred, label) if l != -100]
            for pred, label in zip(all_preds, all_labels)
        ]
        true_labels = [
            [id2tag[l] for (p, l) in zip(pred, label) if l != -100]
            for pred, label in zip(all_preds, all_labels)
        ]
        
        accuracy = sum(p == l for pred, label in zip(true_predictions, true_labels)
                       for p, l in zip(pred, label)) / sum(len(l) for l in true_labels)
        f1 = f1_score(true_labels, true_predictions)
        
        return float(total_loss / len(self.test_loader)), len(self.test_loader.dataset),
               {"accuracy": accuracy, "f1": f1}

In [ ]:
# Cell 7: Run DP-FedAvg with different epsilon values
epsilon_configs = [
    {"noise_multiplier": 2.0, "label": "High Privacy (ε ≈ 1)"},
    {"noise_multiplier": 1.1, "label": "Medium Privacy (ε ≈ 3)"},
    {"noise_multiplier": 0.5, "label": "Low Privacy (ε ≈ 8)"},
]

dp_results = {}

for config in epsilon_configs:
    print(f"\n{'='*50}")
    print(f"Running: {config['label']}")
    print(f"{'='*50}")
    
    def dp_client_fn(cid: str):
        client_id = int(cid)
        return DPBanglaBERTClient(
            client_id,
            client_train_loaders[client_id],
            client_test_loaders[client_id],
            device,
            noise_multiplier=config['noise_multiplier'],
        )
    
    dp_strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=3,
        min_evaluate_clients=3,
        min_available_clients=3,
    )
    
    dp_history = start_simulation(
        client_fn=dp_client_fn,
        num_clients=3,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=dp_strategy,
        client_resources={"num_gpus": 1.0, "num_cpus": 2.0},
    )
    
    dp_results[config['label']] = dp_history
    
    # Save
    fname = config['label'].replace(' ', '_').replace('(', '').replace(')', '').replace('≈', '')
    with open(f"/content/FedMed-BN/results/dp_{fname}_history.pkl", 'wb') as f:
        pickle.dump(dp_history, f)

In [ ]:
# Cell 8: Compare all results
import pandas as pd

def extract_final_metrics(history, name):
    if not history.metrics_distributed:
        return {}
    acc = history.metrics_distributed.get('accuracy', [])
    f1 = history.metrics_distributed.get('f1', [])
    return {
        'method': name,
        'final_accuracy': acc[-1][1] if acc else 0,
        'final_f1': f1[-1][1] if f1 else 0,
    }

all_results = []
all_results.append(extract_final_metrics(history, "FedAvg (No DP)"))

for label, hist in dp_results.items():
    all_results.append(extract_final_metrics(hist, f"DP-FedAvg ({label})"))

df = pd.DataFrame(all_results)
print(df.to_string(index=False))

df.to_csv('/content/FedMed-BN/results/comparison_table.csv', index=False)

# Privacy-Utility Trade-off Plot
epsilons = [1, 3, 8, float('inf')]  # inf = no DP
f1_scores = [r['final_f1'] for r in all_results]

plt.figure(figsize=(8, 5))
plt.plot(epsilons[:-1], f1_scores[:-1], 'o-', label='With DP')
plt.axhline(y=f1_scores[-1], color='r', linestyle='--', label='No DP (Centralized)')
plt.xscale('log')
plt.xlabel('Privacy Budget (ε)')
plt.ylabel('F1 Score')
plt.title('Privacy-Utility Trade-off')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/content/FedMed-BN/results/privacy_utility_tradeoff.png')
plt.show()